# City-Service Review Profiles: Google Colab ML Compute Engine
### High-Performance Clustering & PCA Backend

Pre-configured with your static domain: `doctrine-wages-hubcap.ngrok-free.dev`

### Step 1: Install Dependencies

In [ ]:
import io
import json
from typing import Optional, Dict, Any
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

# Your Credentials
NGROK_AUTHTOKEN = "3IXML8C4lGPfsP7ehgRNGLoE5OC_4mb2Hu7PwsmABx1EbtvQ1"
STATIC_DOMAIN = "doctrine-wages-hubcap.ngrok-free.dev"
PORT = 8000

app = FastAPI(title="Google Colab ML Engine", version="2.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/")
@app.get("/health")
def health():
    return {"status": "online", "worker": "colab-ml-engine", "domain": STATIC_DOMAIN}

def parse_dataset_bytes(file_bytes: bytes, filename: str = "") -> pd.DataFrame:
    fname = (filename or "").lower()
    if fname.endswith((".json", ".jsonl")):
        try:
            return pd.read_json(io.BytesIO(file_bytes), lines=True)
        except Exception:
            pass
        try:
            return pd.read_json(io.BytesIO(file_bytes))
        except Exception:
            pass
        try:
            text = file_bytes.decode("utf-8", errors="ignore")
            records = [json.loads(line) for line in text.splitlines() if line.strip()]
            if records:
                return pd.DataFrame(records)
        except Exception as e:
            raise ValueError(f"Could not parse JSON dataset: {e}")
        raise ValueError("JSON dataset contains no valid records.")
    elif fname.endswith(".parquet"):
        return pd.read_parquet(io.BytesIO(file_bytes))
    else:
        try:
            return pd.read_csv(io.BytesIO(file_bytes))
        except Exception:
            try:
                return pd.read_csv(io.BytesIO(file_bytes), sep=None, engine="python")
            except Exception:
                try:
                    return pd.read_json(io.BytesIO(file_bytes), lines=True)
                except Exception:
                    return pd.read_json(io.BytesIO(file_bytes))

@app.post("/train-clustering")
async def train_clustering(file: UploadFile = File(...), k: Optional[int] = Form(None), k_query: Optional[int] = Query(None, alias="k")):
    resolved_k = max(2, min(10, k or k_query or 4))
    contents = await file.read()
    try:
        df = parse_dataset_bytes(contents, file.filename or "")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Could not parse file: {e}")

    col_map = {str(c).strip().lower(): c for c in df.columns}
    for c in ["avg_rating", "rating", "mean_rating", "num_of_reviews", "review_count", "user_ratings_total", "price_level", "latitude", "longitude"]:
        if c in col_map and col_map[c] in df.columns:
            df[col_map[c]] = pd.to_numeric(df[col_map[c]], errors="coerce")

    if "gmap_id" in col_map and "rating" in col_map:
        gmap_col, rating_col = col_map["gmap_id"], col_map["rating"]
        df_valid = df.dropna(subset=[gmap_col, rating_col]).copy()
        text_col = next((col_map[c] for c in ["text", "review_text", "review", "comment"] if c in col_map), None)
        df_valid["_review_length"] = df_valid[text_col].fillna("").astype(str).str.len() if text_col else 0.0
        grouped = df_valid.groupby(gmap_col)
        mean_rating = grouped[rating_col].mean().rename("mean_rating")
        std_rating = grouped[rating_col].std(ddof=0).fillna(0.0).rename("std_rating")
        log_review_count = np.log1p(grouped.size()).rename("log_review_count")
        avg_review_length = grouped["_review_length"].mean().rename("avg_review_length")
        features_df = pd.concat([mean_rating, std_rating, log_review_count, avg_review_length], axis=1).reset_index()
        biz_ids = features_df[gmap_col].astype(str)
        features_df = features_df[["mean_rating", "std_rating", "log_review_count", "avg_review_length"]]
        schema_case = "Case A: Raw Reviews Aggregation"
    else:
        id_col = next((col_map[c] for c in ["gmap_id", "business_id", "id", "name"] if c in col_map), None)
        biz_ids = df[id_col].astype(str) if id_col else pd.Series([f"item_{i+1}" for i in range(len(df))])
        numeric_df = df.drop(columns=[id_col]).select_dtypes(include=[np.number]) if id_col else df.select_dtypes(include=[np.number])
        count_col = next((col_map[c] for c in ["num_of_reviews", "review_count", "user_ratings_total"] if c in col_map and c in numeric_df.columns), None)
        if count_col and "log_review_count" not in numeric_df.columns:
            numeric_df["log_review_count"] = np.log1p(numeric_df[count_col].clip(lower=0))
        features_df = numeric_df.fillna(numeric_df.median(numeric_only=True)).dropna(axis=1)
        biz_ids = biz_ids.loc[features_df.index]
        schema_case = "Case B: Tabular Numeric Features"

    n_samples = len(features_df)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features_df.values)
    kmeans = KMeans(n_clusters=resolved_k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_scaled)

    sil_score = float(silhouette_score(X_scaled, labels)) if len(np.unique(labels)) > 1 and n_samples > resolved_k else 0.0
    db_index = float(davies_bouldin_score(X_scaled, labels)) if len(np.unique(labels)) > 1 else 0.0
    inertia_val = float(kmeans.inertia_)
    pca = PCA(n_components=2, random_state=42)
    pca_coords = pca.fit_transform(X_scaled)
    explained_var = [float(round(v, 4)) for v in pca.explained_variance_ratio_]

    features_with_cluster = features_df.copy()
    features_with_cluster["cluster"] = labels
    profiles = []
    for c_id in range(resolved_k):
        c_df = features_with_cluster[features_with_cluster["cluster"] == c_id]
        p_dict = {"cluster": int(c_id), "size": len(c_df), "percentage": round((len(c_df) / n_samples) * 100.0, 2)}
        for col in features_df.columns:
            p_dict[col] = round(float(c_df[col].mean()) if len(c_df) > 0 else 0.0, 4)
        profiles.append(p_dict)

    id_list = biz_ids.tolist()
    MAX_SCATTER = 5000
    sample_indices = np.linspace(0, n_samples - 1, MAX_SCATTER, dtype=int) if n_samples > MAX_SCATTER else range(n_samples)
    scatter_points = [{"id": str(id_list[i]), "pca_x": round(float(pca_coords[i, 0]), 4), "pca_y": round(float(pca_coords[i, 1]), 4), "cluster": int(labels[i])} for i in sample_indices]
    centroids_pca = pca.transform(kmeans.cluster_centers_)
    centroids = [{"cluster": int(c), "pca_x": round(float(centroids_pca[c, 0]), 4), "pca_y": round(float(centroids_pca[c, 1]), 4)} for c in range(resolved_k)]

    return {
        "status": "success",
        "sample_size": int(n_samples),
        "k_clusters": int(resolved_k),
        "schema_detected": schema_case,
        "feature_names": list(features_df.columns),
        "metrics": {"silhouette_score": round(sil_score, 4), "davies_bouldin_index": round(db_index, 4), "inertia": round(inertia_val, 2), "pca_variance_ratio": explained_var},
        "profiles": profiles,
        "scatter_points": scatter_points,
        "centroids": centroids
    }

# Start Ngrok static tunnel
nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTHTOKEN)

try:
    tunnel = ngrok.connect(addr=PORT, domain=STATIC_DOMAIN)
except Exception:
    tunnel = ngrok.connect(addr=PORT)

public_url = tunnel.public_url.replace("http://", "https://")
print("\n" + "="*65)
print("🚀 GOOGLE COLAB ML TRAINING ENGINE IS LIVE!")
print(f"👉 STATIC DOMAIN: {public_url}")
print(f"👉 HEALTH PROBE: {public_url}/health")
print("="*65 + "\n")

uvicorn.run(app, host="0.0.0.0", port=PORT)



### Step 2: Start Google Colab ML Training Engine
Run this cell to start FastAPI on port 8000 and bind to `doctrine-wages-hubcap.ngrok-free.dev`.

In [ ]:
import io
from typing import Optional, Dict, Any
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

# Your Credentials
NGROK_AUTHTOKEN = "3IXML8C4lGPfsP7ehgRNGLoE5OC_4mb2Hu7PwsmABx1EbtvQ1"
STATIC_DOMAIN = "doctrine-wages-hubcap.ngrok-free.dev"
PORT = 8000

app = FastAPI(title="Google Colab ML Engine", version="2.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/")
@app.get("/health")
def health():
    return {"status": "online", "worker": "colab-ml-engine", "domain": STATIC_DOMAIN}

@app.post("/train-clustering")
async def train_clustering(file: UploadFile = File(...), k: Optional[int] = Form(None), k_query: Optional[int] = Query(None, alias="k")):
    resolved_k = max(2, min(10, k or k_query or 4))
    contents = await file.read()
    try:
        if (file.filename or "").lower().endswith(".json"):
            df = pd.read_json(io.BytesIO(contents))
        else:
            df = pd.read_csv(io.BytesIO(contents))
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Could not parse file: {e}")

    col_map = {str(c).strip().lower(): c for c in df.columns}
    if "gmap_id" in col_map and "rating" in col_map:
        gmap_col, rating_col = col_map["gmap_id"], col_map["rating"]
        df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
        df_valid = df.dropna(subset=[gmap_col, rating_col]).copy()
        text_col = next((col_map[c] for c in ["text", "review_text", "review", "comment"] if c in col_map), None)
        df_valid["_review_length"] = df_valid[text_col].fillna("").astype(str).str.len() if text_col else 0.0
        grouped = df_valid.groupby(gmap_col)
        mean_rating = grouped[rating_col].mean().rename("mean_rating")
        std_rating = grouped[rating_col].std(ddof=0).fillna(0.0).rename("std_rating")
        log_review_count = np.log1p(grouped.size()).rename("log_review_count")
        avg_review_length = grouped["_review_length"].mean().rename("avg_review_length")
        features_df = pd.concat([mean_rating, std_rating, log_review_count, avg_review_length], axis=1).reset_index()
        biz_ids = features_df[gmap_col].astype(str)
        features_df = features_df[["mean_rating", "std_rating", "log_review_count", "avg_review_length"]]
        schema_case = "Case A: Raw Reviews Aggregation"
    else:
        id_col = next((col_map[c] for c in ["gmap_id", "business_id", "id", "name"] if c in col_map), None)
        biz_ids = df[id_col].astype(str) if id_col else pd.Series([f"item_{i+1}" for i in range(len(df))])
        numeric_df = df.drop(columns=[id_col]).select_dtypes(include=[np.number]) if id_col else df.select_dtypes(include=[np.number])
        features_df = numeric_df.fillna(numeric_df.median(numeric_only=True)).dropna(axis=1)
        schema_case = "Case B: Tabular Numeric Features"

    n_samples = len(features_df)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features_df.values)
    kmeans = KMeans(n_clusters=resolved_k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_scaled)

    sil_score = float(silhouette_score(X_scaled, labels)) if len(np.unique(labels)) > 1 and n_samples > resolved_k else 0.0
    db_index = float(davies_bouldin_score(X_scaled, labels)) if len(np.unique(labels)) > 1 else 0.0
    inertia_val = float(kmeans.inertia_)
    pca = PCA(n_components=2, random_state=42)
    pca_coords = pca.fit_transform(X_scaled)
    explained_var = [float(round(v, 4)) for v in pca.explained_variance_ratio_]

    features_with_cluster = features_df.copy()
    features_with_cluster["cluster"] = labels
    profiles = []
    for c_id in range(resolved_k):
        c_df = features_with_cluster[features_with_cluster["cluster"] == c_id]
        p_dict = {"cluster": int(c_id), "size": len(c_df), "percentage": round((len(c_df) / n_samples) * 100.0, 2)}
        for col in features_df.columns:
            p_dict[col] = round(float(c_df[col].mean()) if len(c_df) > 0 else 0.0, 4)
        profiles.append(p_dict)

    id_list = biz_ids.tolist()
    scatter_points = [{"id": str(id_list[i]), "pca_x": round(float(pca_coords[i, 0]), 4), "pca_y": round(float(pca_coords[i, 1]), 4), "cluster": int(labels[i])} for i in range(min(n_samples, 5000))]
    centroids_pca = pca.transform(kmeans.cluster_centers_)
    centroids = [{"cluster": int(c), "pca_x": round(float(centroids_pca[c, 0]), 4), "pca_y": round(float(centroids_pca[c, 1]), 4)} for c in range(resolved_k)]

    return {
        "status": "success",
        "sample_size": int(n_samples),
        "k_clusters": int(resolved_k),
        "schema_detected": schema_case,
        "feature_names": list(features_df.columns),
        "metrics": {"silhouette_score": round(sil_score, 4), "davies_bouldin_index": round(db_index, 4), "inertia": round(inertia_val, 2), "pca_variance_ratio": explained_var},
        "profiles": profiles,
        "scatter_points": scatter_points,
        "centroids": centroids
    }

# Start Ngrok static tunnel
nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTHTOKEN)

try:
    tunnel = ngrok.connect(addr=PORT, domain=STATIC_DOMAIN)
except Exception:
    tunnel = ngrok.connect(addr=PORT)

public_url = tunnel.public_url.replace("http://", "https://")
print("\n" + "="*65)
print("🚀 GOOGLE COLAB ML TRAINING ENGINE IS LIVE!")
print(f"👉 STATIC DOMAIN: {public_url}")
print(f"👉 HEALTH PROBE: {public_url}/health")
print("="*65 + "\n")

uvicorn.run(app, host="0.0.0.0", port=PORT)